[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/04_colab_standard_comparison.ipynb)

# SensorTwin — multi-seed model comparison at `colab_standard`

Runs the full slate (logreg / random_forest / xgboost / cnn / lstm / **SensorPatchTST**) at
`colab_standard` (20k samples) over several seeds, aggregates to **mean ± std**, and significance-tests
the transformer against the strongest baseline. This answers the project's headline question — *does
the transformer overtake the baselines at scale, and on which classes?* — with the same statistics the
agentic reviewer uses (`sensortwin/evaluation/statistics.py`).

Runtime: **Runtime → Change runtime type → GPU** first. ~30–60 min on a T4 for 3 seeds. Synthetic
benchmark — a controlled testbed, not real-world validation (see `reports/model_card.md`).

In [ ]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — runs on CPU (much slower at 20k; AMP stays off).")

In [ ]:
# --- run configuration (edit here) ---
MODELS = "logreg,random_forest,xgboost,cnn,lstm,transformer"
SEEDS = [0, 1, 2]            # mean +/- std over independent runs (data + split + init per seed)
EPOCHS = 15                  # deep-model epochs; matches the single-seed transformer run
MODE = "colab_standard"      # 20k samples; use quick_demo (2k) for a fast dry run
OUTROOT = "runs"
BASELINES = {"logreg", "random_forest", "xgboost", "cnn", "lstm"}
MODEL_LIST = MODELS.split(",")

In [ ]:
# --- run the full slate once per seed (reuses the audited train_baseline orchestration) ---
import time
from scripts.train_baseline import main as train_baseline

for s in SEEDS:
    t0 = time.time()
    print(f"=== seed {s} ===")
    train_baseline([
        "--mode", MODE, "--epochs", str(EPOCHS), "--models", MODELS,
        "--seed", str(s), "--out", f"{OUTROOT}/seed{s}", "--no-anomaly",
    ])
    print(f"  seed {s} done in {time.time() - t0:.0f}s")

In [ ]:
# --- aggregate per-seed metrics -> mean +/- std + significance (the proper log) ---
import json
from pathlib import Path

from sensortwin.evaluation.statistics import compare_seeds, mean_std


def _load(seed, model):
    p = Path(f"{OUTROOT}/seed{seed}/metrics_{model}.json")
    return json.loads(p.read_text()) if p.exists() else None


agg = {m: {"macro_f1": [], "macro_auroc": [], "ece": [], "per_class": []} for m in MODEL_LIST}
for m in MODEL_LIST:
    for s in SEEDS:
        d = _load(s, m)
        if d is None:
            continue
        agg[m]["macro_f1"].append(d["macro_f1"])
        agg[m]["macro_auroc"].append(d.get("macro_auroc"))
        agg[m]["ece"].append(d.get("ece"))
        agg[m]["per_class"].append(d["per_class"])
present = [m for m in MODEL_LIST if agg[m]["macro_f1"]]


def _fmt(vals):
    vals = [v for v in vals if v is not None]
    if not vals:
        return "—"
    mu, sd = mean_std(vals)
    return f"{mu:.3f} ± {sd:.3f}"


rows = sorted(present, key=lambda m: -mean_std(agg[m]["macro_f1"])[0])
table = ["| Model | Macro-F1 | Macro-AUROC | ECE |", "| --- | ---: | ---: | ---: |"]
for m in rows:
    table.append(f"| {m} | {_fmt(agg[m]['macro_f1'])} | {_fmt(agg[m]['macro_auroc'])} | {_fmt(agg[m]['ece'])} |")
table = "\n".join(table)

verdict, per_class_md = "", ""
if "transformer" in present:
    bbs = [m for m in present if m in BASELINES]
    if bbs:
        best = max(bbs, key=lambda m: mean_std(agg[m]["macro_f1"])[0])
        cmp = compare_seeds(agg[best]["macro_f1"], agg["transformer"]["macro_f1"])
        kind = "beats" if (cmp.significant and cmp.delta > 0) else (
            "loses to" if (cmp.significant and cmp.delta < 0) else "ties")
        verdict = (
            f"**Transformer vs best baseline ({best}):** {cmp.delta:+.3f} macro-F1 "
            f"(95% CI [{cmp.ci_low:.3f}, {cmp.ci_high:.3f}], p={cmp.p_value:.3f}) — "
            f"the transformer **{kind}** the strongest baseline over {cmp.n_seeds} seeds."
        )
        classes = list(agg["transformer"]["per_class"][0].keys())
        cm = lambda model, c: sum(pc[c]["f1"] for pc in agg[model]["per_class"]) / len(agg[model]["per_class"])
        deltas = sorted(((c, cm("transformer", c) - cm(best, c)) for c in classes), key=lambda kv: -kv[1])
        per_class_md = "\n".join(
            ["", f"Per-class F1 delta (transformer − {best}), best/worst:", ""]
            + [f"- `{c}`: {d:+.3f}" for c, d in deltas[:3] + deltas[-3:]]
        )

summary_md = (
    f"# colab_standard comparison ({len(SEEDS)} seeds, {EPOCHS} epochs)\n\n"
    f"{table}\n\n{verdict}\n{per_class_md}\n"
)
Path(f"{OUTROOT}/summary.md").write_text(summary_md)
Path(f"{OUTROOT}/summary.json").write_text(json.dumps(
    {m: {k: agg[m][k] for k in ("macro_f1", "macro_auroc", "ece")} for m in present}, indent=2))

print(summary_md)
print(f"saved {OUTROOT}/summary.md and {OUTROOT}/summary.json")

In [ ]:
# --- download the results (Colab is ephemeral) ---
try:
    from google.colab import files
    !zip -qr runs.zip {OUTROOT}
    files.download(f"{OUTROOT}/summary.md")
    files.download(f"{OUTROOT}/summary.json")
    files.download("runs.zip")
except Exception as e:
    print("Not in Colab or download unavailable:", e)
    print(f"Results are in ./{OUTROOT}/ (summary.md, summary.json, per-seed metrics).")

## What next

- **Paste `summary.md` back into the chat** — it'll become the README *Results* headline (mean ± std)
  and a research-log entry, with the honest framing (which model wins at scale, on which classes,
  and whether the transformer's edge over the best baseline is statistically significant).
- Heavier studies at this scale (separate notebooks/commands): masked-pretraining label efficiency
  (`scripts.label_efficiency_sweep`), robustness/calibration (`scripts.robustness_report`), and the
  agentic ablation runner (`scripts.run_agent`).

These are research-grade only at `colab_standard`+; the synthetic numbers are a controlled testbed,
not real-world validation.